In [3]:


import os, json, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY", "").strip()   # .strip() wegen des Leerzeichen-Problems vom letzten Lab
if not api_key:
    raise ValueError("OPENAI_API_KEY not found – liegt die .env im selben Ordner wie das Notebook?")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

def call_openai(prompt, temperature=0.7, max_tokens=500, system=None, json_mode=False):
    """Ein einzelner API-Call, gibt den Antworttext zurück."""
    messages = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": prompt}]
    kwargs = dict(model=MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens)
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    try:
        resp = client.chat.completions.create(**kwargs)
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {type(e).__name__}: {e}"

def run_multiple(prompt, n=5, max_workers=5, **kwargs):
    """Schickt denselben Prompt n-mal parallel ab."""
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        return list(ex.map(lambda _: call_openai(prompt, **kwargs), range(n)))

def analyze_consistency(responses, normalize=lambda r: r.strip().lower()):
    """Konsistenz = Anteil der häufigsten (normalisierten) Antwort."""
    normed = [normalize(r) for r in responses]
    counts = Counter(normed)
    top, top_n = counts.most_common(1)[0]
    return {
        "runs": len(responses),
        "unique_responses": len(counts),
        "consistency_%": round(100 * top_n / len(responses), 1),
        "most_common": top[:80],
        "avg_length_chars": round(sum(len(r) for r in responses) / len(responses)),
        "errors": sum(r.startswith("ERROR") for r in responses),
    }

# Checkpoint
test = call_openai("Say 'Setup successful!' and nothing else.", temperature=0)
print(test)
print("✅ Setup complete" if "successful" in test.lower() else "⚠️ Setup prüfen")


Setup successful!
✅ Setup complete


## Part 1 – Step 2: Initial Zero-Shot Prompts (v1 / Baseline)
Intentionally minimal prompts without any format or content requirements, to reproduce the issues of TechFlow's current chatbot.

In [2]:
# --- v1 prompts (baseline) ---
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

v1_prompts = {
    "Sentiment Analysis": sentiment_prompt_v1,
    "Product Description": product_prompt_v1,
    "Data Extraction": extraction_prompt_v1,
}

# Test each prompt once
for name, prompt in v1_prompts.items():
    result = call_openai(prompt)          # temperature=0.7 (default)
    print(f"{'='*60}\n{name} – Result:\n{'='*60}")
    print(result, "\n")

Sentiment Analysis – Result:
The customer message can be classified as **Positive Feedback**. 

Product Description – Result:
**Product Name: SwiftClick Wireless Mouse**

**Price: $29.99**

**Description:**

Elevate your computing experience with the SwiftClick Wireless Mouse, designed for seamless navigation and ultimate comfort. This sleek, modern mouse combines advanced technology with an ergonomic design, making it the perfect companion for both work and play.

**Key Features:**

- **Wireless Freedom:** Enjoy the convenience of a reliable 2.4GHz wireless connection with a range of up to 33 feet, eliminating tangled cords and providing you the freedom to move around your workspace.

- **Precision Tracking:** The SwiftClick features a high-precision optical sensor with adjustable DPI settings (800, 1200, 1600) to suit your preferences, ensuring smooth scrolling and accurate tracking on various surfaces.

- **Ergonomic Design:** Crafted with comfort in mind, its contoured shape fits s

## Part 2 – Steps 3–5: Diagnosing Failures (5, 10, 15 runs)

Exact string matching is meaningless for generative tasks (every text differs), so each task gets a **signature** (what *should* be consistent) and a **target spec** (what a production system needs). The same evaluation is reused for v2 and v3.

| Task | Signature (consistency) | Target spec (format adherence) |
|---|---|---|
| Sentiment | extracted label (positive / negative / neutral / mixed) | response is exactly one lowercase label |
| Product description | length bucket + markdown yes/no | 60–100 words, plain text, no invented numbers/specs |
| Data extraction | set of JSON keys (or "not-json") | valid JSON with exactly the keys `order_id`, `order_date`, `delivery_speed`, `packaging_condition` |

In [4]:
import re, statistics

# ---------- Sentiment ----------
def sentiment_signature(r):
    m = re.search(r"\b(positive|negative|neutral|mixed)\b", r.lower())
    return m.group(1) if m else "unclear"

def sentiment_format_ok(r):
    return r.strip() in {"positive", "negative", "neutral"}

# ---------- Product description ----------
def word_count(r):
    return len(r.split())

def has_markdown(r):
    return bool(re.search(r"\*\*|^#|^\s*[-*] ", r, re.M))

def has_invented_numbers(r):
    """True if the text contains any number other than the given price 29.99."""
    nums = re.findall(r"\d+(?:[.,]\d+)?", r)
    return any(n not in {"29.99", "29,99"} for n in nums)

def product_signature(r):
    wc = word_count(r)
    bucket = "short(<60)" if wc < 60 else "target(60-100)" if wc <= 100 else "long(>100)"
    return f"{bucket} | {'markdown' if has_markdown(r) else 'plain'}"

def product_format_ok(r):
    return 60 <= word_count(r) <= 100 and not has_markdown(r) and not has_invented_numbers(r)

# ---------- Data extraction ----------
EXPECTED_KEYS = {"order_id", "order_date", "delivery_speed", "packaging_condition"}

def parse_json(r):
    """Try the whole response first, then the outermost {...} block (e.g. after CoT reasoning)."""
    text = re.sub(r"```(?:json)?", "", r).strip()
    for candidate in (text, (re.search(r"\{.*\}", text, re.S) or [None])[0]):
        if candidate:
            try:
                return json.loads(candidate)
            except json.JSONDecodeError:
                pass
    return None

def extraction_signature(r):
    d = parse_json(r)
    return "keys: " + ", ".join(sorted(d)) if isinstance(d, dict) else "not-json"

def extraction_format_ok(r):
    d = parse_json(r)
    return isinstance(d, dict) and set(d) == EXPECTED_KEYS

# ---------- Generic evaluation ----------
TASK_EVAL = {
    "Sentiment Analysis":  {"signature": sentiment_signature,  "format_ok": sentiment_format_ok},
    "Product Description": {"signature": product_signature,    "format_ok": product_format_ok,
                            "extra": {"invented_numbers_%": has_invented_numbers}},
    "Data Extraction":     {"signature": extraction_signature, "format_ok": extraction_format_ok},
}

def evaluate(task, responses):
    ev = TASK_EVAL[task]
    sigs = [ev["signature"](r) for r in responses]
    top_sig, top_n = Counter(sigs).most_common(1)[0]
    row = {
        "runs": len(responses),
        "unique_raw_responses": len(set(r.strip() for r in responses)),
        "unique_signatures": len(set(sigs)),
        "consistency_%": round(100 * top_n / len(responses), 1),
        "format_adherence_%": round(100 * sum(ev["format_ok"](r) for r in responses) / len(responses), 1),
        "avg_words": round(statistics.mean(word_count(r) for r in responses)),
        "most_common_signature": top_sig,
        "errors": sum(r.startswith("ERROR") for r in responses),
    }
    for name, fn in ev.get("extra", {}).items():
        row[name] = round(100 * sum(fn(r) for r in responses) / len(responses), 1)
    return row

os.makedirs("results", exist_ok=True)
all_runs = {}   # (version, task, n) -> list of responses

def run_experiment(version, prompts, ns=(5, 10, 15), **call_kwargs):
    rows = []
    for task, prompt in prompts.items():
        for n in ns:
            responses = run_multiple(prompt, n=n, **call_kwargs)
            all_runs[(version, task, n)] = responses
            rows.append({"version": version, "task": task, **evaluate(task, responses)})
    # save raw evidence
    with open(f"results/{version}_runs.json", "w") as f:
        json.dump({f"{t} | n={n}": r for (v, t, n), r in all_runs.items() if v == version}, f, indent=2)
    return pd.DataFrame(rows)

print("✅ Evaluation helpers ready")

✅ Evaluation helpers ready


In [5]:
df_v1 = run_experiment("v1", v1_prompts)   # temperature=0.7 (default)
pd.set_option("display.max_colwidth", 60)
df_v1

,version,task,runs,unique_raw_responses,unique_signatures,consistency_%,format_adherence_%,avg_words,most_common_signature,errors,invented_numbers_%
0,v1,Sentiment Analysis,5,5,1,100.0,0.0,12,positive,0,NaN
1,v1,Sentiment Analysis,10,5,1,100.0,0.0,12,positive,0,NaN
2,v1,Sentiment Analysis,15,11,1,100.0,0.0,11,positive,0,NaN
3,v1,Product Description,5,5,1,100.0,0.0,291,long(>100) | markdown,0,100.0
4,v1,Product Description,10,10,1,100.0,0.0,286,long(>100) | markdown,0,100.0
5,v1,Product Description,15,15,1,100.0,0.0,297,long(>100) | markdown,0,93.3
6,v1,Data Extraction,5,5,1,100.0,0.0,22,not-json,0,NaN
7,v1,Data Extraction,10,7,1,100.0,0.0,20,not-json,0,NaN
8,v1,Data Extraction,15,13,1,100.0,0.0,24,not-json,0,NaN


In [6]:
# --- Step 5: Fine-grained diagnostics on the raw v1 responses ---
# The coarse signatures show the bot is "consistently wrong". These metrics reveal the hidden variation.

def sentiment_label_variants(r):
    """Label phrases the model invented (in **bold** or "quotes"), e.g. ('customer satisfaction', 'positive feedback')."""
    found = [a or b for a, b in re.findall(r'\*\*(.+?)\*\*|"(.+?)"', r)]
    if not found:
        found = re.findall(r"as (.+?)\.?$", r)
    return tuple(sorted(set(x.lower().strip(" .") for x in found)))

def extraction_field_names(r):
    """Field names used in a bullet-list answer, e.g. ('delivery speed', 'order date', ...)."""
    return tuple(sorted(f.strip().lower() for f in re.findall(r"^\s*-\s*\**([^:*\n]+?)\**\s*:", r, re.M)))

diag_rows = []
for n in (5, 10, 15):
    s = all_runs[("v1", "Sentiment Analysis", n)]
    p = all_runs[("v1", "Product Description", n)]
    e = all_runs[("v1", "Data Extraction", n)]
    sl, ef = Counter(map(sentiment_label_variants, s)), Counter(map(extraction_field_names, e))
    wc = [word_count(r) for r in p]
    diag_rows.append({
        "runs": n,
        "sentiment_label_variants": len(sl),
        "sentiment_top_label_%": round(100 * sl.most_common(1)[0][1] / n),
        "sentiment_multi_label_answers": sum(len(sentiment_label_variants(r)) > 1 for r in s),
        "product_words_min–max": f"{min(wc)}–{max(wc)}",
        "product_words_std": round(statistics.pstdev(wc)),
        "product_invented_name_SwiftClick": sum("swiftclick" in r.lower() for r in p),
        "extraction_field_sets": len(ef),
        "extraction_top_field_set_%": round(100 * ef.most_common(1)[0][1] / n),
        "extraction_with_intro_text": sum(r.lower().startswith("here") for r in e),
    })

df_diag_v1 = pd.DataFrame(diag_rows).set_index("runs")
display(df_diag_v1)

print("\nSentiment label variants (15 runs):")
for k, v in Counter(map(sentiment_label_variants, all_runs[("v1", "Sentiment Analysis", 15)])).most_common():
    print(f"  {v}x  {k}")
print("\nExtraction field sets (15 runs):")
for k, v in Counter(map(extraction_field_names, all_runs[("v1", "Data Extraction", 15)])).most_common():
    print(f"  {v}x  {k}")

,sentiment_label_variants,sentiment_top_label_%,sentiment_multi_label_answers,product_words_min–max,product_words_std,product_invented_name_SwiftClick,extraction_field_sets,extraction_top_field_set_%,extraction_with_intro_text
runs,,,,,,,,,
5,3,60,4,264–339,26,0,4,40,3
10,3,70,7,219–331,35,2,4,70,4
15,6,40,9,259–343,26,6,6,53,11



Sentiment label variants (15 runs):
  6x  ('customer satisfaction', 'positive feedback')
  5x  ('positive feedback',)
  1x  ('positive feedback', 'satisfaction')
  1x  ('positive feedback', 'positive review')
  1x  ('positive feedback or a positive review',)
  1x  ('positive feedback', 'satisfaction feedback')

Extraction field sets (15 runs):
  8x  ('delivery speed', 'order date', 'order number', 'packaging condition')
  2x  ('delivery', 'order date', 'order number', 'packaging condition')
  2x  ('delivery speed', 'issue', 'order date', 'order number')
  1x  ('delivery', 'item ordered', 'order date', 'packaging condition')
  1x  ('delivery speed', 'order date', 'order item number', 'packaging condition')
  1x  ('delivery experience', 'order date', 'order number', 'packaging condition')


### Step 5 – Failure Analysis (v1, 15 runs, temperature 0.7)

**Key insight:** The coarse signatures show 100 % consistency but 0 % format adherence – the bot is *consistently wrong*. The fine-grained diagnostics reveal the hidden variation, and it grows with more runs (sentiment label variants: 3 → 3 → 6; extraction field sets: 4 → 4 → 6 for 5 → 10 → 15 runs).

| Task | Format adherence | Failure patterns observed | Business impact | Fix planned |
|---|---|---|---|---|
| Sentiment Analysis | 0 % | Full sentence instead of a label; markdown/quotes; invented label names ("Positive Feedback", "Customer Satisfaction", "Positive Review"); 9/15 answers give **two** labels; 6 different label combinations | Responses cannot be parsed or aggregated automatically (e.g. counting negative tickets) | v2: fixed label set + single-word constraint; v3: few-shot examples |
| Product Description | 0 % | 219–343 words (target 60–100); always markdown with headings/bullets; **hallucinated specs** in almost every run (battery 15/15, AA 13/15, 2.4 GHz 11/15, DPI 7/15) and invented product name "SwiftClick" (6/15) | Cannot be pasted into a shop template; false product claims → legal/returns risk | v2: length, style and "only given facts" constraints; v3: few-shot + fixed structure |
| Data Extraction | 0 % (never JSON) | Bullet list instead of JSON; 6 different field-name sets ("Order Number" / "Item ordered" / "Order Item Number", "Delivery" / "Delivery Speed" / "Delivery Experience", "Issue"); ID with/without "#"; date not normalised, no year; 11/15 with intro sentence | Cannot be loaded into a database/CRM without manual cleaning | v2: JSON schema with fixed keys; v3: Chain-of-Thought + JSON mode |

## Part 3 – Steps 6–8: Iteration 1 (v2) – Clear Instructions, Format & Constraints

Changes compared to v1 (still zero-shot, no examples yet):
- **Role & clear task** – the model knows what it is and what exactly to do
- **Explicit output format** – fixed label set / word range / JSON schema
- **Constraints** – single word, plain text, "use only the given facts", no extra text

Temperature stays at **0.7** (same as v1), so any improvement comes from the prompt alone.

In [7]:
# --- Step 6: Sentiment v2 ---
sentiment_prompt_v2 = """
You are a sentiment classifier for a customer service system.

Classify the sentiment of the customer message below as exactly ONE of these labels:
positive, negative, neutral

Rules:
- Respond with the label only, in lowercase.
- No punctuation, no quotes, no explanation.

Customer message: "I love this product! It's exactly what I needed."
Sentiment:
"""

# --- Step 7: Product description v2 ---
product_prompt_v2 = """
You are a copywriter for an online electronics shop.

Write a product description using ONLY the facts below.
Product facts:
- Product: wireless mouse
- Price: $29.99

Requirements:
- Length: 60-100 words.
- Structure: one hook sentence, 2-3 sentences about everyday benefits, one closing sentence that mentions the price.
- Style: friendly and professional, address the reader as "you".
- Plain text only: no markdown, no headings, no bullet points, no product name.
- Do NOT invent technical specifications or numbers (e.g. DPI, battery life, range, Bluetooth/2.4 GHz). Describe benefits in general terms instead.
- Write the price exactly as "$29.99".
"""

# --- Step 8: Data extraction v2 ---
extraction_prompt_v2 = """
Extract information from the customer feedback below and return it as a JSON object.

Use exactly these keys:
- "order_id": the order/item number as a string, without "#" (e.g. "12345")
- "order_date": the order date as written in the text, or null if not mentioned
- "delivery_speed": one of "fast", "on_time", "slow", or null if not mentioned
- "packaging_condition": one of "good", "damaged", or null if not mentioned

Return ONLY the JSON object – no explanation, no markdown code fences.

Customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

v2_prompts = {
    "Sentiment Analysis": sentiment_prompt_v2,
    "Product Description": product_prompt_v2,
    "Data Extraction": extraction_prompt_v2,
}

# Additional quality check for extraction: are the extracted VALUES correct (not just the keys)?
EXPECTED_VALUES = {"order_id": "12345", "delivery_speed": "fast", "packaging_condition": "damaged"}

def extraction_values_correct(r):
    d = parse_json(r)
    return isinstance(d, dict) and all(str(d.get(k, "")).lower() == v for k, v in EXPECTED_VALUES.items())

TASK_EVAL["Data Extraction"]["extra"] = {"correct_values_%": extraction_values_correct}

# Quick single test before the full run
for name, prompt in v2_prompts.items():
    print(f"{'='*60}\n{name} – v2 single test:\n{'='*60}\n{call_openai(prompt)}\n")

Sentiment Analysis – v2 single test:
positive

Product Description – v2 single test:
Upgrade your workspace with this sleek wireless mouse that combines style and functionality. You’ll enjoy the freedom of movement without the hassle of tangled wires, making it perfect for both home and office use. Its ergonomic design ensures comfort during long hours of work or gaming, enhancing your productivity. Grab yours today for just $29.99 and experience the convenience of wireless technology!

Data Extraction – v2 single test:
{
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}



In [8]:
df_v2 = run_experiment("v2", v2_prompts)   # temperature=0.7, same as v1
display(df_v2)

# v1 vs v2 at 15 runs
cols = ["version", "task", "consistency_%", "format_adherence_%", "unique_signatures",
        "unique_raw_responses", "avg_words", "invented_numbers_%", "correct_values_%"]
comparison_v1_v2 = pd.concat([df_v1, df_v2]).query("runs == 15").reindex(columns=cols)
comparison_v1_v2

,version,task,runs,unique_raw_responses,unique_signatures,consistency_%,format_adherence_%,avg_words,most_common_signature,errors,invented_numbers_%,correct_values_%
0,v2,Sentiment Analysis,5,1,1,100.0,100.0,1,positive,0,NaN,NaN
1,v2,Sentiment Analysis,10,1,1,100.0,100.0,1,positive,0,NaN,NaN
2,v2,Sentiment Analysis,15,1,1,100.0,100.0,1,positive,0,NaN,NaN
3,v2,Product Description,5,5,2,80.0,20.0,57,short(<60) | plain,0,0.0,NaN
4,v2,Product Description,10,10,2,70.0,30.0,57,short(<60) | plain,0,0.0,NaN
5,v2,Product Description,15,15,2,60.0,40.0,58,short(<60) | plain,0,0.0,NaN
6,v2,Data Extraction,5,1,1,100.0,100.0,11,"keys: delivery_speed, order_date, order_id, packaging_co...",0,NaN,100.0
7,v2,Data Extraction,10,1,1,100.0,100.0,11,"keys: delivery_speed, order_date, order_id, packaging_co...",0,NaN,100.0
8,v2,Data Extraction,15,1,1,100.0,100.0,11,"keys: delivery_speed, order_date, order_id, packaging_co...",0,NaN,100.0


,version,task,consistency_%,format_adherence_%,unique_signatures,unique_raw_responses,avg_words,invented_numbers_%,correct_values_%
2,v1,Sentiment Analysis,100.0,0.0,1,11,11,NaN,NaN
5,v1,Product Description,100.0,0.0,1,15,297,93.3,NaN
8,v1,Data Extraction,100.0,0.0,1,13,24,NaN,NaN
2,v2,Sentiment Analysis,100.0,100.0,1,1,1,NaN,NaN
5,v2,Product Description,60.0,40.0,2,15,58,0.0,NaN
8,v2,Data Extraction,100.0,100.0,1,1,11,NaN,100.0


### v2 Results (15 runs, temperature 0.7)
- **Sentiment & Data Extraction: solved** – format adherence 0 % → 100 %, all 15 responses identical even at temperature 0.7. A clear task leaves the model nothing to "improvise".
- **Product Description: improved but not solved** – invented numbers 93 % → 0 %, no markdown, but format adherence only 40 %:
  - Word counts 48–67 (avg 58): the model undershoots the 60-word minimum → **LLMs cannot count words reliably**.
  - Hidden hallucinations remain: invented qualitative features such as "sleek" (9/15) and "ergonomic design" (3/15) – not caught by the number check.
- **Takeaway:** instructions + format + constraints are enough for classification/extraction; generation needs examples (to calibrate length/style) and a countable structure (sentences instead of words).

## Part 4 – Steps 9–11: Iteration 2 (v3) – Few-Shot & Chain-of-Thought

- **Step 9 – Sentiment: few-shot** (4 examples incl. a neutral question and a mixed praise+complaint message) + explicit business rule for mixed messages
- **Step 10 – Data Extraction: Chain-of-Thought** – the model reasons step by step per field, then outputs the final JSON
- **Step 11 – Product Description: few-shot + structure** – 2 example descriptions (~80 words, other products to avoid copying), exactly 5 sentences, ban on invented features

Prompts are now **templates** (placeholders) so they can be reused on new inputs in Part 5. Temperature stays at 0.7.

In [9]:
# --- Step 9: Sentiment v3 (few-shot) ---
SENTIMENT_TEMPLATE_V3 = """
You are a sentiment classifier for a customer service system.
Classify each customer message as exactly ONE label: positive, negative, neutral

Rules:
- Respond with the label only, in lowercase, no punctuation, no explanation.
- If a message contains any complaint or problem, classify it as negative, even if it also contains praise (complaints need follow-up).
- Factual questions or statements without emotion are neutral.

Examples:
Message: "Absolutely fantastic service, my order arrived a day early!"
Sentiment: positive

Message: "The headphones stopped working after two days. Very disappointed."
Sentiment: negative

Message: "Can you tell me when my order will be shipped?"
Sentiment: neutral

Message: "Great price, but the charger broke after a week."
Sentiment: negative

Message: "{message}"
Sentiment:
"""

# --- Step 10: Data extraction v3 (Chain-of-Thought) ---
EXTRACTION_TEMPLATE_V3 = """
You are a data extraction assistant for customer feedback.
Work through the feedback step by step, then give the final result.

Step 1 - Order ID: find the order or item number. Remove "#". If none is mentioned, use null.
Step 2 - Order date: find the date the order was placed and copy it as written. If none, use null.
Step 3 - Delivery speed: find any statement about delivery time and map it to "fast", "on_time" or "slow". If not mentioned, use null.
Step 4 - Packaging condition: find any statement about the packaging and map it to "good" or "damaged". If not mentioned, use null.

Answer in exactly this format:
Reasoning:
1. Order ID: <short reasoning>
2. Order date: <short reasoning>
3. Delivery speed: <short reasoning>
4. Packaging condition: <short reasoning>
JSON:
{"order_id": ..., "order_date": ..., "delivery_speed": ..., "packaging_condition": ...}

Customer feedback: "{feedback}"
"""

# --- Step 11: Product description v3 (few-shot + structure) ---
PRODUCT_TEMPLATE_V3 = """
You are a copywriter for an online electronics shop.
Write a product description using ONLY the product facts given.

Format rules:
- Exactly 5 sentences in one paragraph, about 80 words in total (60-100 words allowed).
- Sentence 1: hook. Sentences 2-4: everyday benefits. Sentence 5: closing sentence that mentions the price.
- Friendly and professional, address the reader as "you".
- Plain text only: no markdown, no headings, no bullet points, no product name.
- Do NOT invent numbers or technical specs (e.g. DPI, battery life, range, Bluetooth/2.4 GHz).
- Do NOT claim specific features or design details that are not in the facts (e.g. ergonomic shape, sleek design, silent clicks, RGB lighting).
- Write the price exactly as given.

Example 1
Product facts: Product: laptop stand | Price: $39.99
Description: Give your workspace a cleaner, more comfortable setup with this laptop stand. Raising your screen closer to eye level helps you sit more naturally while you work or study every day. It frees up space on your desk, so your keyboard, notes and coffee all have their place. The simple design fits into home offices, shared desks and creative corners alike. Bring a little more order and comfort to your daily routine, wherever you set up, for just $39.99.

Example 2
Product facts: Product: USB-C charging cable | Price: $12.99
Description: Keep your devices powered up without the hassle with this USB-C charging cable. It is a simple, dependable companion for your phone, tablet or other USB-C devices at home or on the go. Toss it into your bag, keep it by your bed or leave it on your desk, so a charge is always within reach. Having a spare cable means no more searching or sharing when your battery runs low. Stay connected and ready for your day for just $12.99.

Now write the description.
Product facts: Product: {product} | Price: {price}
Description:
"""

# Template builders (.replace instead of .format, because the JSON braces would break .format)
def build_sentiment_prompt(message):
    return SENTIMENT_TEMPLATE_V3.replace("{message}", message)

def build_extraction_prompt(feedback):
    return EXTRACTION_TEMPLATE_V3.replace("{feedback}", feedback)

def build_product_prompt(product, price):
    return PRODUCT_TEMPLATE_V3.replace("{product}", product).replace("{price}", price)

v3_prompts = {
    "Sentiment Analysis": build_sentiment_prompt("I love this product! It's exactly what I needed."),
    "Product Description": build_product_prompt("wireless mouse", "$29.99"),
    "Data Extraction": build_extraction_prompt("I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."),
}

# New check: invented qualitative features (the hidden hallucination found in v2)
UNVERIFIED_FEATURES = ["ergonomic", "sleek", "silent", "rgb", "battery", "dpi", "bluetooth",
                       "rechargeable", "adjustable", "precision", "optical"]

def has_unverified_features(r):
    return any(w in r.lower() for w in UNVERIFIED_FEATURES)

TASK_EVAL["Product Description"]["extra"] = {
    "invented_numbers_%": has_invented_numbers,
    "unverified_features_%": has_unverified_features,
}

# Quick single test
for name, prompt in v3_prompts.items():
    print(f"{'='*60}\n{name} – v3 single test:\n{'='*60}\n{call_openai(prompt)}\n")

Sentiment Analysis – v3 single test:
positive

Product Description – v3 single test:
Experience the freedom of movement with this wireless mouse. Say goodbye to tangled cords and enjoy a clutter-free workspace while you navigate your tasks with ease. It's perfect for both work and play, allowing you to switch seamlessly between applications or enjoy your favorite games without interruption. Lightweight and portable, you can take it anywhere, ensuring you stay productive on the go. Elevate your computing experience for just $29.99.

Data Extraction – v3 single test:
Reasoning:
1. Order ID: The order number is mentioned as #12345, so I remove the "#" to get 12345.
2. Order date: The date when the order was placed is given as March 15th.
3. Delivery speed: The feedback states that the delivery was "fast."
4. Packaging condition: The feedback mentions that the packaging was "damaged."

JSON:
{"order_id": 12345, "order_date": "March 15th", "delivery_speed": "fast", "packaging_condition": "d

In [10]:
df_v3 = run_experiment("v3", v3_prompts)   # temperature=0.7, same as v1 and v2
display(df_v3)

# Re-evaluate ALL versions from the stored responses, so the new check (unverified_features_%) applies to v1 and v2 too
def comparison_table(versions, n=15):
    rows = [{"version": v, "task": t, **evaluate(t, all_runs[(v, t, n)])}
            for t in TASK_EVAL for v in versions]
    return pd.DataFrame(rows).reindex(columns=cols + ["unverified_features_%"])

comparison_v1_v2_v3 = comparison_table(["v1", "v2", "v3"])
comparison_v1_v2_v3

,version,task,runs,unique_raw_responses,unique_signatures,consistency_%,format_adherence_%,avg_words,most_common_signature,errors,invented_numbers_%,unverified_features_%,correct_values_%
0,v3,Sentiment Analysis,5,1,1,100.0,100.0,1,positive,0,NaN,NaN,NaN
1,v3,Sentiment Analysis,10,1,1,100.0,100.0,1,positive,0,NaN,NaN,NaN
2,v3,Sentiment Analysis,15,1,1,100.0,100.0,1,positive,0,NaN,NaN,NaN
3,v3,Product Description,5,5,1,100.0,100.0,71,target(60-100) | plain,0,0.0,0.0,NaN
4,v3,Product Description,10,10,1,100.0,100.0,74,target(60-100) | plain,0,0.0,0.0,NaN
5,v3,Product Description,15,15,1,100.0,100.0,73,target(60-100) | plain,0,0.0,0.0,NaN
6,v3,Data Extraction,5,5,1,100.0,100.0,77,"keys: delivery_speed, order_date, order_id, packaging_co...",0,NaN,NaN,100.0
7,v3,Data Extraction,10,10,1,100.0,100.0,72,"keys: delivery_speed, order_date, order_id, packaging_co...",0,NaN,NaN,100.0
8,v3,Data Extraction,15,15,1,100.0,100.0,70,"keys: delivery_speed, order_date, order_id, packaging_co...",0,NaN,NaN,100.0


,version,task,consistency_%,format_adherence_%,unique_signatures,unique_raw_responses,avg_words,invented_numbers_%,correct_values_%,unverified_features_%
0,v1,Sentiment Analysis,100.0,0.0,1,11,11,NaN,NaN,NaN
1,v2,Sentiment Analysis,100.0,100.0,1,1,1,NaN,NaN,NaN
2,v3,Sentiment Analysis,100.0,100.0,1,1,1,NaN,NaN,NaN
3,v1,Product Description,100.0,0.0,1,15,297,93.3,NaN,100.0
4,v2,Product Description,60.0,40.0,2,15,58,0.0,NaN,66.7
5,v3,Product Description,100.0,100.0,1,15,73,0.0,NaN,0.0
6,v1,Data Extraction,100.0,0.0,1,13,24,NaN,0.0,NaN
7,v2,Data Extraction,100.0,100.0,1,1,11,NaN,100.0,NaN
8,v3,Data Extraction,100.0,100.0,1,15,70,NaN,100.0,NaN


### v3 Results (15 runs, temperature 0.7)
- **Product Description: solved** – format adherence 40 % → 100 %. All 15 texts have exactly 5 sentences and 66–81 words; unverified features ("sleek", "ergonomic") 67 % → 0 %. No phrases were copied from the examples – few-shot transferred length, structure and tone, not wording.
  - Remaining limitation: some texts still claim soft features such as "reliable connectivity" or "portable design". A keyword list cannot catch every hallucination → would need an LLM-as-judge or human review.
- **Sentiment: stays at 100 %** – the value of the examples shows on harder inputs (Part 5).
- **Data Extraction: hidden regression found.** The table shows 100 %, but in 13/15 runs `order_id` is the **number** `12345` instead of the **string** `"12345"` (v2 was correct). Cause: the CoT output template `{"order_id": ..., ...}` lost the type hint, and the check used `str()` so it could not see the difference. In production this breaks IDs with leading zeros ("00123" → 123) and typed database columns.
  → Fix: stricter check (types) + v3.1 prompt with an explicit typed output template.

In [11]:
# Stricter checks: values must be strings (or null) and must match exactly – no str() conversion
def extraction_format_ok(r):
    d = parse_json(r)
    if not (isinstance(d, dict) and set(d) == EXPECTED_KEYS):
        return False
    return all(d[k] is None or isinstance(d[k], str) for k in EXPECTED_KEYS)

def extraction_values_correct(r):
    d = parse_json(r)
    return isinstance(d, dict) and all(d.get(k) == v for k, v in EXPECTED_VALUES.items())

# TASK_EVAL still points to the old functions, so re-register them
TASK_EVAL["Data Extraction"]["format_ok"] = extraction_format_ok
TASK_EVAL["Data Extraction"]["extra"] = {"correct_values_%": extraction_values_correct}

# Make the comparison skip versions that don't contain every task (needed for v3.1 below)
def comparison_table(versions, n=15):
    rows = [{"version": v, "task": t, **evaluate(t, all_runs[(v, t, n)])}
            for t in TASK_EVAL for v in versions if (v, t, n) in all_runs]
    return pd.DataFrame(rows).reindex(columns=cols + ["unverified_features_%"])

print("order_id types in v3 (15 runs):",
      Counter(type(parse_json(r)["order_id"]).__name__ for r in all_runs[("v3", "Data Extraction", 15)]))
comparison_table(["v1", "v2", "v3"]).query("task == 'Data Extraction'")

order_id types in v3 (15 runs): Counter({'int': 13, 'str': 2})


,version,task,consistency_%,format_adherence_%,unique_signatures,unique_raw_responses,avg_words,invented_numbers_%,correct_values_%,unverified_features_%
6,v1,Data Extraction,100.0,0.0,1,13,24,NaN,0.0,NaN
7,v2,Data Extraction,100.0,100.0,1,1,11,NaN,100.0,NaN
8,v3,Data Extraction,100.0,13.3,1,15,70,NaN,13.3,NaN


In [12]:
# --- Step 10 (fix): Data extraction v3.1 – CoT with an explicit, typed output template ---
EXTRACTION_TEMPLATE_V3_1 = """
You are a data extraction assistant for customer feedback.
Work through the feedback step by step, then give the final result.

Step 1 - Order ID: find the order or item number. Remove "#" and keep it as a string (e.g. "12345"). If none is mentioned, use null.
Step 2 - Order date: find the date the order was placed and copy it as written. If none, use null.
Step 3 - Delivery speed: find any statement about delivery time and map it to "fast", "on_time" or "slow". If not mentioned, use null.
Step 4 - Packaging condition: find any statement about the packaging and map it to "good" or "damaged". If not mentioned, use null.

Answer in exactly this format:
Reasoning:
1. Order ID: <short reasoning>
2. Order date: <short reasoning>
3. Delivery speed: <short reasoning>
4. Packaging condition: <short reasoning>
JSON:
{"order_id": "<string or null>", "order_date": "<string or null>", "delivery_speed": "<fast | on_time | slow | null>", "packaging_condition": "<good | damaged | null>"}

Every value in the JSON must be a string in double quotes or null – never a bare number.

Customer feedback: "{feedback}"
"""

def build_extraction_prompt(feedback):          # overrides the v3 builder → used in Part 5
    return EXTRACTION_TEMPLATE_V3_1.replace("{feedback}", feedback)

df_v3_1 = run_experiment("v3.1", {
    "Data Extraction": build_extraction_prompt(
        "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.")
})
display(df_v3_1)

print("order_id types in v3.1 (15 runs):",
      Counter(type(parse_json(r)["order_id"]).__name__ for r in all_runs[("v3.1", "Data Extraction", 15)]))
comparison_table(["v1", "v2", "v3", "v3.1"])

,version,task,runs,unique_raw_responses,unique_signatures,consistency_%,format_adherence_%,avg_words,most_common_signature,errors,correct_values_%
0,v3.1,Data Extraction,5,5,1,100.0,100.0,69,"keys: delivery_speed, order_date, order_id, packaging_co...",0,100.0
1,v3.1,Data Extraction,10,10,1,100.0,100.0,73,"keys: delivery_speed, order_date, order_id, packaging_co...",0,100.0
2,v3.1,Data Extraction,15,15,1,100.0,100.0,71,"keys: delivery_speed, order_date, order_id, packaging_co...",0,100.0


order_id types in v3.1 (15 runs): Counter({'str': 15})


,version,task,consistency_%,format_adherence_%,unique_signatures,unique_raw_responses,avg_words,invented_numbers_%,correct_values_%,unverified_features_%
0,v1,Sentiment Analysis,100.0,0.0,1,11,11,NaN,NaN,NaN
1,v2,Sentiment Analysis,100.0,100.0,1,1,1,NaN,NaN,NaN
2,v3,Sentiment Analysis,100.0,100.0,1,1,1,NaN,NaN,NaN
3,v1,Product Description,100.0,0.0,1,15,297,93.3,NaN,100.0
4,v2,Product Description,60.0,40.0,2,15,58,0.0,NaN,66.7
5,v3,Product Description,100.0,100.0,1,15,73,0.0,NaN,0.0
6,v1,Data Extraction,100.0,0.0,1,13,24,NaN,0.0,NaN
7,v2,Data Extraction,100.0,100.0,1,1,11,NaN,100.0,NaN
8,v3,Data Extraction,100.0,13.3,1,15,70,NaN,13.3,NaN
9,v3.1,Data Extraction,100.0,100.0,1,15,71,NaN,100.0,NaN


## Part 5 – Step 12: Task Variations

So far each prompt was tested on a single input. Here the best prompts are tested on **new, harder inputs with known correct answers** (5 runs each), measuring **accuracy** in addition to format and consistency.

- **Sentiment (v2 vs v3):** clear negative, neutral question, mixed praise + complaint, sarcasm, neutral statement, German message, lukewarm review
- **Product description (v3):** other products, including one where a feature word appears in the product name and one with given numbers (allowed numbers must not be flagged)
- **Data extraction (v2 vs v3.1):** leading-zero order ID, missing fields (→ null), no order info at all, German feedback, ISO date + vague wording ("slightly torn")

In [13]:
N_VAR = 5
variation_runs = {}   # raw evidence for Part 5

def build_sentiment_prompt_v2(message):
    return sentiment_prompt_v2.split("Customer message:")[0] + f'Customer message: "{message}"\nSentiment:\n'

sentiment_cases = [
    ("The package arrived three days late and nobody answered my emails.", "negative"),   # clear negative
    ("Do you ship to Austria?", "neutral"),                                                # question
    ("The mouse works great, but the delivery took two weeks.", "negative"),              # mixed → business rule
    ("Oh great, another cable that stopped working after one day. Just perfect.", "negative"),  # sarcasm
    ("I received my order today.", "neutral"),                                             # factual statement
    ("Thanks for the quick refund, really appreciated!", "positive"),                      # clear positive
    ("Das Produkt ist super, bin sehr zufrieden!", "positive"),                            # German
    ("It's okay, does what it says.", "neutral"),                                          # lukewarm / ambiguous
]

rows = []
for msg, expected in sentiment_cases:
    for version, builder in [("v2", build_sentiment_prompt_v2), ("v3", build_sentiment_prompt)]:
        resp = run_multiple(builder(msg), n=N_VAR)
        variation_runs[f"sentiment | {version} | {msg}"] = resp
        labels = [r.strip().lower() for r in resp]
        top_n = Counter(labels).most_common(1)[0][1]
        rows.append({"message": msg[:50], "expected": expected, "version": version,
                     "answers": dict(Counter(labels)),
                     "accuracy_%": 100 * sum(l == expected for l in labels) / N_VAR,
                     "consistency_%": 100 * top_n / N_VAR})

df_sent_var = pd.DataFrame(rows)
display(df_sent_var)
display(df_sent_var.groupby("version")[["accuracy_%", "consistency_%"]].mean().round(1))

,message,expected,version,answers,accuracy_%,consistency_%
0,The package arrived three days late and nobody ans,negative,v2,{'negative': 5},100.0,100.0
1,The package arrived three days late and nobody ans,negative,v3,{'negative': 5},100.0,100.0
2,Do you ship to Austria?,neutral,v2,{'neutral': 5},100.0,100.0
3,Do you ship to Austria?,neutral,v3,{'neutral': 5},100.0,100.0
4,"The mouse works great, but the delivery took two w",negative,v2,"{'neutral': 3, 'mixed': 2}",0.0,60.0
5,"The mouse works great, but the delivery took two w",negative,v3,{'negative': 5},100.0,100.0
6,"Oh great, another cable that stopped working after",negative,v2,{'negative': 5},100.0,100.0
7,"Oh great, another cable that stopped working after",negative,v3,{'negative': 5},100.0,100.0
8,I received my order today.,neutral,v2,{'neutral': 5},100.0,100.0
9,I received my order today.,neutral,v3,{'neutral': 5},100.0,100.0


,accuracy_%,consistency_%
version,,
v2,87.5,95.0
v3,100.0,100.0


In [14]:
product_cases = [
    ("noise-cancelling headphones", "$89.99"),
    ("ergonomic office chair", "$249.00"),                                # feature word in the product name → allowed
    ("portable Bluetooth speaker with 12-hour battery life", "$59.99"),  # given numbers → allowed
]

def product_checks(r, product, price):
    facts = f"{product} {price}".lower()
    allowed_nums = set(re.findall(r"\d+(?:[.,]\d+)?", facts))
    return {
        "words_60_100": 60 <= word_count(r) <= 100,
        "five_sentences": len(re.findall(r"[.!?](?:\s|$)", r)) == 5,
        "plain_text": not has_markdown(r),
        "price_exact": price in r,
        "no_invented_numbers": set(re.findall(r"\d+(?:[.,]\d+)?", r)) <= allowed_nums,
        "no_unverified_features": not any(w in r.lower() and w not in facts for w in UNVERIFIED_FEATURES),
    }

rows = []
for product, price in product_cases:
    resp = run_multiple(build_product_prompt(product, price), n=N_VAR)
    variation_runs[f"product | v3 | {product}"] = resp
    checks = [product_checks(r, product, price) for r in resp]
    row = {"product": product, "avg_words": round(statistics.mean(word_count(r) for r in resp))}
    for k in checks[0]:
        row[f"{k}_%"] = 100 * sum(c[k] for c in checks) / N_VAR
    row["format_adherence_%"] = 100 * sum(all(c.values()) for c in checks) / N_VAR
    rows.append(row)

df_prod_var = pd.DataFrame(rows)
df_prod_var

,product,avg_words,words_60_100_%,five_sentences_%,plain_text_%,price_exact_%,no_invented_numbers_%,no_unverified_features_%,format_adherence_%
0,noise-cancelling headphones,73,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1,ergonomic office chair,72,100.0,100.0,100.0,100.0,100.0,40.0,40.0
2,portable Bluetooth speaker with 12-hour battery life,80,100.0,100.0,100.0,100.0,100.0,100.0,100.0


In [15]:
def build_extraction_prompt_v2(feedback):
    return extraction_prompt_v2.split("Customer feedback:")[0] + f'Customer feedback: "{feedback}"\n'

NULLS = dict.fromkeys(EXPECTED_KEYS)
extraction_cases = [
    ("Order #00789 arrived on time and the box was in perfect condition. I placed it on June 2nd.",
     {"order_id": "00789", "order_date": "June 2nd", "delivery_speed": "on_time", "packaging_condition": "good"}),
    ("My package took almost three weeks to arrive. Order number 55821.",
     {**NULLS, "order_id": "55821", "delivery_speed": "slow"}),
    ("I'm really happy with the headphones, the sound is amazing!",
     NULLS),
    ("Bestellung #4471 vom 3. Mai kam schnell an, aber der Karton war eingedrückt.",
     {"order_id": "4471", "order_date": "3. Mai", "delivery_speed": "fast", "packaging_condition": "damaged"}),
    ("Ordered on 2024-01-10 (order 98765). Delivery was quick, packaging slightly torn.",
     {"order_id": "98765", "order_date": "2024-01-10", "delivery_speed": "fast", "packaging_condition": "damaged"}),
]

rows = []
for feedback, expected in extraction_cases:
    for version, builder in [("v2", build_extraction_prompt_v2), ("v3.1", build_extraction_prompt)]:
        resp = run_multiple(builder(feedback), n=N_VAR)
        variation_runs[f"extraction | {version} | {feedback}"] = resp
        parsed = [parse_json(r) or {} for r in resp]
        field_acc = [sum(d.get(k) == v for k, v in expected.items()) / len(expected) for d in parsed]
        wrong = Counter(f"{k}={d.get(k)!r}" for d in parsed for k, v in expected.items() if d.get(k) != v)
        rows.append({"feedback": feedback[:45], "version": version,
                     "format_ok_%": 100 * sum(extraction_format_ok(r) for r in resp) / N_VAR,
                     "field_accuracy_%": round(100 * statistics.mean(field_acc), 1),
                     "all_fields_correct_%": 100 * sum(a == 1 for a in field_acc) / N_VAR,
                     "wrong_values": dict(wrong)})

df_extr_var = pd.DataFrame(rows)
display(df_extr_var)
display(df_extr_var.groupby("version")[["format_ok_%", "field_accuracy_%", "all_fields_correct_%"]].mean().round(1))

with open("results/variations_runs.json", "w") as f:
    json.dump(variation_runs, f, indent=2, ensure_ascii=False)
print("✅ Saved results/variations_runs.json")

,feedback,version,format_ok_%,field_accuracy_%,all_fields_correct_%,wrong_values
0,Order #00789 arrived on time and the box was,v2,100.0,100.0,100.0,{}
1,Order #00789 arrived on time and the box was,v3.1,100.0,100.0,100.0,{}
2,My package took almost three weeks to arrive.,v2,100.0,100.0,100.0,{}
3,My package took almost three weeks to arrive.,v3.1,100.0,100.0,100.0,{}
4,"I'm really happy with the headphones, the sou",v2,100.0,100.0,100.0,{}
5,"I'm really happy with the headphones, the sou",v3.1,100.0,100.0,100.0,{}
6,"Bestellung #4471 vom 3. Mai kam schnell an, a",v2,100.0,100.0,100.0,{}
7,"Bestellung #4471 vom 3. Mai kam schnell an, a",v3.1,100.0,100.0,100.0,{}
8,Ordered on 2024-01-10 (order 98765). Delivery,v2,100.0,100.0,100.0,{}
9,Ordered on 2024-01-10 (order 98765). Delivery,v3.1,100.0,100.0,100.0,{}


,format_ok_%,field_accuracy_%,all_fields_correct_%
version,,,
v2,100.0,100.0,100.0
v3.1,100.0,100.0,100.0


✅ Saved results/variations_runs.json


In [16]:
for r in variation_runs["product | v3 | ergonomic office chair"]:
    flagged = [w for w in UNVERIFIED_FEATURES if w in r.lower() and w not in "ergonomic office chair"]
    print(flagged, "|", r, "\n")

['adjustable'] | Transform your work-from-home experience with this ergonomic office chair. It provides the support you need to stay comfortable during long hours of sitting, helping to reduce fatigue and improve your focus. With adjustable features, you can customize it to fit your unique body shape and preferences, promoting better posture and well-being. This chair is designed to enhance your productivity while seamlessly fitting into your workspace. Elevate your comfort and efficiency for just $249.00. 

['adjustable'] | Transform your work environment with this ergonomic office chair. Designed to provide support during long hours, it helps reduce discomfort and fatigue, allowing you to focus on your tasks. The adjustable features ensure a personalized fit, promoting better posture and productivity throughout your day. Whether you're working from home or in an office, this chair enhances your workspace comfort. Invest in your well-being and elevate your work experience for just $24

### Step 12 – Results: What works and what needs adjustment

| Task | Result on variations | What works | What needs adjustment |
|---|---|---|---|
| Sentiment | v2: 87.5 % accuracy / 95 % consistency → **v3: 100 % / 100 %** | Both versions handle sarcasm, German, questions and lukewarm reviews | v2 fails the **mixed** message (3× neutral, 2× "mixed" – a label that was not even allowed). The few-shot example + business rule in v3 fixes it → examples matter most for edge cases |
| Product description | Headphones & speaker 100 %, **office chair 40 %** | Length, 5 sentences, plain text, exact price; given numbers ("12-hour") correctly kept | Chair: "adjustable" invented in 3/5 texts (+ unchecked health claims like "reduce fatigue"). The ban list was mouse-specific → the model fills in features **typical for the category**. → v3.2: general rule "only features literally in the facts" |
| Data extraction | **v2 and v3.1 both 100 %** on all 5 cases | Leading zeros, nulls, no order info, German input, vague wording ("slightly torn") | Nothing for accuracy – but CoT costs ~6× more output (≈71 vs 11 words) and caused the type regression. For simple field extraction, v2 is the better production choice; CoT pays off for multi-step reasoning |

In [17]:
# --- Tuning: Product description v3.2 – general rule instead of a mouse-specific ban list ---
OLD_RULE = "- Do NOT claim specific features or design details that are not in the facts (e.g. ergonomic shape, sleek design, silent clicks, RGB lighting)."
NEW_RULE = ("- Mention ONLY features that are literally stated in the product facts. Do NOT assume features that are "
            "typical for this kind of product (e.g. adjustable, waterproof, ergonomic, sleek, silent, RGB) unless they "
            "appear in the facts, and make no health claims. Describe general everyday benefits instead.")

assert OLD_RULE in PRODUCT_TEMPLATE_V3, "Rule text not found – check the template"
PRODUCT_TEMPLATE_V3_2 = PRODUCT_TEMPLATE_V3.replace(OLD_RULE, NEW_RULE)

def build_product_prompt(product, price):          # overrides the v3 builder → used for the final evaluation
    return PRODUCT_TEMPLATE_V3_2.replace("{product}", product).replace("{price}", price)

def evaluate_product_cases(version, cases, n=N_VAR):
    rows = []
    for product, price in cases:
        resp = run_multiple(build_product_prompt(product, price), n=n)
        variation_runs[f"product | {version} | {product}"] = resp
        checks = [product_checks(r, product, price) for r in resp]
        row = {"version": version, "product": product,
               "avg_words": round(statistics.mean(word_count(r) for r in resp))}
        for k in checks[0]:
            row[f"{k}_%"] = 100 * sum(c[k] for c in checks) / n
        row["format_adherence_%"] = 100 * sum(all(c.values()) for c in checks) / n
        rows.append(row)
    return pd.DataFrame(rows)

df_prod_var_v3_2 = evaluate_product_cases("v3.2", product_cases)
display(pd.concat([df_prod_var.assign(version="v3"), df_prod_var_v3_2])[
    ["version", "product", "avg_words", "no_unverified_features_%", "format_adherence_%"]])

for r in variation_runs["product | v3.2 | ergonomic office chair"]:
    print("-", r[:160], "...\n")

with open("results/variations_runs.json", "w") as f:
    json.dump(variation_runs, f, indent=2, ensure_ascii=False)

,version,product,avg_words,no_unverified_features_%,format_adherence_%
0,v3,noise-cancelling headphones,73,100.0,100.0
1,v3,ergonomic office chair,72,40.0,40.0
2,v3,portable Bluetooth speaker with 12-hour battery life,80,100.0,100.0
0,v3.2,noise-cancelling headphones,73,100.0,100.0
1,v3.2,ergonomic office chair,72,60.0,60.0
2,v3.2,portable Bluetooth speaker with 12-hour battery life,77,100.0,100.0


- Transform your workspace with this ergonomic office chair that prioritizes your comfort. Enjoy long hours of productivity without the strain, as this chair supp ...

- Transform your workspace with this ergonomic office chair designed for your comfort. You can enjoy long hours of productivity without discomfort, making your wo ...

- Transform your work environment with this ergonomic office chair. Designed to support you during long hours of sitting, it enhances your comfort and productivit ...

- Transform your work experience with this ergonomic office chair. Designed to provide support during long hours at your desk, it helps you maintain better postur ...

- Transform your workday experience with this ergonomic office chair. Designed to provide support during those long hours at your desk, it helps you maintain comf ...



## Part 5 – Step 13: Final Evaluation & Comparison

Final prompts, each run 15 times on the original inputs (temperature 0.7):
- Sentiment Analysis → **v3** (few-shot + business rule)
- Product Description → **v3.2** (few-shot + 5-sentence structure + "only given facts" rule)
- Data Extraction → **v3.1** (Chain-of-Thought + typed JSON template)

In [18]:
final_prompts = {
    "Sentiment Analysis":  build_sentiment_prompt("I love this product! It's exactly what I needed."),       # v3
    "Product Description": build_product_prompt("wireless mouse", "$29.99"),                                # v3.2
    "Data Extraction":     build_extraction_prompt("I ordered item #12345 on March 15th. "
                                                   "The delivery was fast but the packaging was damaged."), # v3.1
}

df_final = run_experiment("final", final_prompts, ns=(15,))
display(df_final)

# Full comparison across all iterations (15 runs each)
report = comparison_table(["v1", "v2", "v3", "v3.1", "final"])
display(report)

# Improvement summary v1 → final
summary = (report[report["version"].isin(["v1", "final"])]
           .pivot(index="task", columns="version", values=["format_adherence_%", "consistency_%", "avg_words"]))
display(summary)

# Success criterion from the rubric: >80 % consistency across 15 runs
for _, row in df_final.iterrows():
    ok = row["consistency_%"] > 80 and row["format_adherence_%"] > 80
    print(f"{'✅' if ok else '❌'} {row['task']}: consistency {row['consistency_%']} %, "
          f"format adherence {row['format_adherence_%']} %")

report.to_csv("results/final_comparison.csv", index=False)
print("\n✅ Saved results/final_comparison.csv")

,version,task,runs,unique_raw_responses,unique_signatures,consistency_%,format_adherence_%,avg_words,most_common_signature,errors,invented_numbers_%,unverified_features_%,correct_values_%
0,final,Sentiment Analysis,15,1,1,100.0,100.0,1,positive,0,NaN,NaN,NaN
1,final,Product Description,15,15,1,100.0,100.0,72,target(60-100) | plain,0,0.0,6.7,NaN
2,final,Data Extraction,15,15,1,100.0,100.0,75,"keys: delivery_speed, order_date, order_id, packaging_co...",0,NaN,NaN,100.0


,version,task,consistency_%,format_adherence_%,unique_signatures,unique_raw_responses,avg_words,invented_numbers_%,correct_values_%,unverified_features_%
0,v1,Sentiment Analysis,100.0,0.0,1,11,11,NaN,NaN,NaN
1,v2,Sentiment Analysis,100.0,100.0,1,1,1,NaN,NaN,NaN
2,v3,Sentiment Analysis,100.0,100.0,1,1,1,NaN,NaN,NaN
3,final,Sentiment Analysis,100.0,100.0,1,1,1,NaN,NaN,NaN
4,v1,Product Description,100.0,0.0,1,15,297,93.3,NaN,100.0
5,v2,Product Description,60.0,40.0,2,15,58,0.0,NaN,66.7
6,v3,Product Description,100.0,100.0,1,15,73,0.0,NaN,0.0
7,final,Product Description,100.0,100.0,1,15,72,0.0,NaN,6.7
8,v1,Data Extraction,100.0,0.0,1,13,24,NaN,0.0,NaN
9,v2,Data Extraction,100.0,100.0,1,1,11,NaN,100.0,NaN


format_adherence_%      consistency_%        avg_words  \
version                          final   v1         final     v1     final   
task                                                                         
Data Extraction                  100.0  0.0         100.0  100.0      75.0   
Product Description              100.0  0.0         100.0  100.0      72.0   
Sentiment Analysis               100.0  0.0         100.0  100.0       1.0   

                            
version                 v1  
task                        
Data Extraction       24.0  
Product Description  297.0  
Sentiment Analysis    11.0

✅ Sentiment Analysis: consistency 100.0 %, format adherence 100.0 %
✅ Product Description: consistency 100.0 %, format adherence 100.0 %
✅ Data Extraction: consistency 100.0 %, format adherence 100.0 %

✅ Saved results/final_comparison.csv


### Step 13 – Final Results & Conclusion

| Task | Final prompt | Format adherence v1 → final | Avg words v1 → final | Key technique |
|---|---|---|---|---|
| Sentiment Analysis | v3 | 0 % → **100 %** | 11 → 1 | fixed label set (v2); few-shot + business rule for edge cases (v3) |
| Product Description | v3.2 | 0 % → **100 %** | 297 → 72 | few-shot examples + 5-sentence structure + "only given facts" rule |
| Data Extraction | v3.1 | 0 % → **100 %** | 24 → 75 | JSON schema (v2); CoT + typed output template (v3.1) |

All final prompts reach **100 % consistency and 100 % format adherence across 15 runs** (target: >80 %).

**Which change had the biggest impact?** v1 → v2 (clear instructions, format, constraints) solved sentiment and extraction completely. Few-shot (v3) was decisive for the generative task (length/structure) and for sentiment edge cases (mixed message: 0 % → 100 % accuracy).

**Few-shot vs. CoT:** Few-shot is best when the *form* or a *judgement rule* must be shown (labels, tone, length). CoT pays off for multi-step reasoning – for simple field extraction it added no accuracy on the variations, cost ~6× more output and introduced a type regression that had to be fixed (v3.1). In production, v2 would be the cheaper choice for this extraction task.

**Remaining limitations:** keyword-based checks only catch anticipated hallucinations (office chair: invented "adjustable", posture claims; 1/15 final mouse texts claims "compact size"). Production would need richer product facts and an LLM-as-judge or human review step.